# 14 — Build v03 model configuration

v03 = v02 + four upgrades, consolidated in one authoritative build notebook. Notebooks 05 (wave coupling) and 13 (tracer setup recipe) are superseded for v03 *building* but remain the reference for methodology.

| Upgrade | Source | Status |
|---|---|---|
| A · Hypersaline initial salinity (42 psu inside, 37.5 outside) | `feedback_salinity_bc.md` | **applied here** |
| B · Passive tracer for residence time | notebook 13 | **applied here** |
| C · ERA5 evaporation flux | literature + Knudsen balance need | stub + download hook |
| D · Extended simulation (≥30 days) | notebook 13 residence requirement | **applied here** — gated on extended BC/meteo |
| E · SWAN wave coupling | notebook 05 | applied here (inlined) |

**Workflow:** copy v02 → v03 cleanly → layer each upgrade → verify.

**Dependencies:** v02 must be complete so we have `v02/output/Stagnone_dxy01_15m_map.nc` (needed only for the face-coordinates used in initial-field XYZ samples).

## 1. Imports and paths

In [ ]:
%matplotlib inline
import os, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import xugrid as xu
import matplotlib.pyplot as plt
from matplotlib.path import Path as MplPath

project_root = Path(r'F:\StagnoneDT')
v02_dir = project_root / 'model' / 'dflowfm_v02'
v03_dir = project_root / 'model' / 'dflowfm_v03'
v02_map = v02_dir / 'output' / 'Stagnone_dxy01_15m_map.nc'

assert v02_dir.exists(), f'v02 missing at {v02_dir}'
print(f'v02 dir:       {v02_dir}')
print(f'v03 dir:       {v03_dir}')
print(f'v02 map.nc:    {v02_map.exists()}')

## 2. Copy v02 → v03

Excludes the large `output/` folder and cache files. Idempotent — overwrites existing files.

In [ ]:
EXCLUDE = {'output', '__pycache__', 'wave'}   # 'wave' excluded; regenerated in section E
EXCLUDE_SUFFIX = {'.cache'}

def copy_tree(src: Path, dst: Path):
    dst.mkdir(parents=True, exist_ok=True)
    n = 0
    for item in src.iterdir():
        if item.name in EXCLUDE or item.suffix in EXCLUDE_SUFFIX:
            continue
        target = dst / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        else:
            shutil.copy2(item, target)
        n += 1
    return n

n = copy_tree(v02_dir, v03_dir)
(v03_dir / 'output').mkdir(exist_ok=True)
print(f'Copied {n} items from v02 to v03.')

## 3. Load face coordinates from v02 mesh

Used for both the salinity and the tracer XYZ sample files. The mesh itself is copied over unchanged.

In [ ]:
if v02_map.exists():
    uds = xu.open_dataset(str(v02_map), chunks={'time': 50})
    fc = uds.grid.face_coordinates
    face_x = np.asarray(fc[:, 0])
    face_y = np.asarray(fc[:, 1])
    print(f'Using face centers from v02 map.nc: {len(face_x)} faces.')
else:
    # Fallback: read face centers directly from _net.nc
    net = xr.open_dataset(str(v03_dir / 'Stagnone_dxy01_15m_net.nc'))
    face_x = net['mesh2d_face_x'].values
    face_y = net['mesh2d_face_y'].values
    print(f'v02 map.nc not available; using face centers from _net.nc: {len(face_x)} faces.')

## A. Hypersaline initial salinity

Write an XYZ sample: 42 psu inside the lagoon polygon, 37.5 outside. Polygon loaded from `data/processed/lagoon_polygon_wgs84_simplified.csv` (82 vertices extracted from the `oldModel/Stagnone_justLagoon` mesh — authoritative hand-drawn lagoon extent).

In [ ]:
poly_csv = project_root / 'data' / 'processed' / 'lagoon_polygon_wgs84_simplified.csv'
poly_df = pd.read_csv(poly_csv)
lagoon_poly = poly_df[['lon', 'lat']].values
lagoon_path = MplPath(lagoon_poly)

inside = lagoon_path.contains_points(np.column_stack([face_x, face_y]))
print(f'Polygon vertices: {len(lagoon_poly)}')
print(f'Lon range: {lagoon_poly[:,0].min():.4f} .. {lagoon_poly[:,0].max():.4f}')
print(f'Lat range: {lagoon_poly[:,1].min():.4f} .. {lagoon_poly[:,1].max():.4f}')
print(f'Interior cells: {inside.sum()} / {len(inside)}  ({100*inside.mean():.1f}%)')

In [ ]:
S_lag = 42.0
S_off = 37.5
sal_vals = np.where(inside, S_lag, S_off)

sal_xyz = v03_dir / 'initialsalinity_hypersaline.xyz'
with open(sal_xyz, 'w') as f:
    for x, y, s in zip(face_x, face_y, sal_vals):
        f.write(f'{x:.6f} {y:.6f} {s:.3f}\n')
print(f'Wrote {sal_xyz}   ({inside.sum()} cells = {S_lag}, {(~inside).sum()} cells = {S_off})')

## B. Passive tracer for residence time

1.0 inside polygon, 0.0 outside, with open-boundary Dirichlet = 0.

In [ ]:
tracer_vals = np.where(inside, 1.0, 0.0)
tracer_xyz = v03_dir / 'lagoon_tracer_init.xyz'
with open(tracer_xyz, 'w') as f:
    for x, y, v in zip(face_x, face_y, tracer_vals):
        f.write(f'{x:.6f} {y:.6f} {v:.3f}\n')
print(f'Wrote {tracer_xyz}')

tracer_bc = v03_dir / 'tracer_zero.bc'
with open(tracer_bc, 'w') as f:
    f.write('[General]\nfileVersion           = 1.01\nfileType              = boundConds\n\n'
            '[Forcing]\nname                  = Stagnone_dxy01_15m_bnd1_0001\n'
            'function              = constant\n'
            'quantity              = tracerbndlagoon_tracer\nunit                  = -\n0.0\n')
print(f'Wrote {tracer_bc}')

## 4. Patch `initialFields.ini`

Append blocks for the hypersaline salinity override and the passive-tracer init. Skip if the blocks are already present (idempotent).

In [ ]:
ini_path = v03_dir / 'initialFields.ini'
existing = ini_path.read_text() if ini_path.exists() else ''

blocks = []
if 'initialsalinity_hypersaline.xyz' not in existing:
    blocks.append(
        '\n[Initial]\n'
        '    quantity              = initialsalinity\n'
        '    dataFile              = initialsalinity_hypersaline.xyz\n'
        '    dataFileType          = sample\n'
        '    interpolationMethod   = averaging\n'
        '    averagingType         = mean\n'
        '    operand               = O\n'
    )
if 'lagoon_tracer_init.xyz' not in existing:
    blocks.append(
        '\n[Initial]\n'
        '    quantity              = initialtracerlagoon_tracer\n'
        '    dataFile              = lagoon_tracer_init.xyz\n'
        '    dataFileType          = sample\n'
        '    interpolationMethod   = averaging\n'
        '    averagingType         = mean\n'
        '    operand               = O\n'
    )

if blocks:
    with open(ini_path, 'a') as f:
        f.writelines(blocks)
    print(f'Appended {len(blocks)} block(s) to {ini_path.name}')
else:
    print('initialFields.ini already has both blocks — no change.')

## 5. Patch MDU — declare the tracer

Add `TracerNames = lagoon_tracer` under the `[physics]` section (D-Flow FM convention). Also double-check `InitialSalinity` scalar is consistent — we'll rely on the `initialFields.ini` override.

In [ ]:
mdu_path = v03_dir / 'Stagnone_dxy01_15m.mdu'
mdu_text = mdu_path.read_text()

if 'TracerNames' not in mdu_text:
    # Insert under [physics]. If section missing, append a [physics] block.
    if '[physics]' in mdu_text.lower():
        import re
        mdu_text = re.sub(
            r'(?i)(\[physics\][^\[]*?)(\n\[)',
            r'\1TracerNames           = lagoon_tracer\n\2',
            mdu_text, count=1,
        )
    else:
        mdu_text += '\n[physics]\nTracerNames           = lagoon_tracer\n'
    mdu_path.write_text(mdu_text)
    print('Added TracerNames = lagoon_tracer to MDU.')
else:
    print('MDU already declares TracerNames — no change.')

## 6. Patch external forcing (`.ext`) — tracer Dirichlet=0

Append a boundary block telling D-Flow FM the tracer arrives at 0 through the open ocean boundary. Uses the same `.pli` as the water-level boundary.

In [ ]:
# Locate the primary .ext file (v02 uses old-format: Stagnone_dxy01_15m_old.ext)
ext_candidates = list(v03_dir.glob('Stagnone_dxy01_15m*.ext'))
print('Found .ext files:', [f.name for f in ext_candidates])
# Prefer the new-style one; otherwise fall back to whatever is there
ext_path = next((f for f in ext_candidates if 'new' in f.name.lower()),
                ext_candidates[0] if ext_candidates else None)
print(f'Editing: {ext_path}')

if ext_path is not None:
    ext_text = ext_path.read_text()
    if 'tracerbndlagoon_tracer' not in ext_text:
        block = (
            '\n[Boundary]\n'
            '    quantity              = tracerbndlagoon_tracer\n'
            '    locationFile          = Stagnone_dxy01_15m.pli\n'
            '    forcingFile           = tracer_zero.bc\n'
        )
        with open(ext_path, 'a') as f:
            f.write(block)
        print('Appended tracer boundary block.')
    else:
        print('Tracer boundary already present — no change.')

## C. ERA5 evaporation — hook only

Download the `e` (evaporation) field for the simulation period. Needed for the Knudsen balance and for a physically honest hypersaline budget. Skipped here if data isn't available — log and continue.

```python
import dfm_tools as dfmt
# dfmt.download_ERA5(varlist=['e'], longitude_min=12.1, longitude_max=12.7,
#                    latitude_min=37.5, latitude_max=38.1,
#                    date_min='2025-07-01', date_max='2025-08-15',
#                    dir_output=str(v03_dir))
```
Then append to the ext file:
```
[Meteo]
    quantity              = rainfall_rate   # or dedicated evaporation quantity
    forcingFile           = era5_e_Stagnone_2025.nc
    forcingFileType       = netcdf
```
**Note:** in D-Flow FM, evaporation is typically negative rainfall or handled via the heat-balance (already on with `Temperature=3`). Verify the current FM release's convention before committing this — it moved between 1.2.x releases.

## D. Extend simulation for residence time

v02 ran 9 days. Residence time in a semi-enclosed lagoon is typically weeks → need ≥30 days for the tracer to decay to ~10%. Change `TStop` in the MDU.

**Blocker:** extending `TStop` requires CMEMS + ERA5 data for the full period. Flag this here; only bump TStop once the extended data is downloaded (notebook 04 has the `dfm_tools.download_*` calls).

In [ ]:
# Show current TStart/TStop for reference. Do NOT bump TStop yet — run short first to verify tracer.
import re
text = mdu_path.read_text()
for key in ['RefDate', 'TStart', 'TStop', 'Tunit']:
    m = re.search(rf'(?im)^{key}\s*=\s*(\S+)', text)
    print(f'{key:10s} = {m.group(1) if m else "(not found)"}')

In [ ]:
# When ready to extend: uncomment and set the new TStop (in minutes relative to RefDate)
# EXTEND_TO_MINUTES = 60*24*45   # 45 days
# text = re.sub(r'(?im)^(TStop\s*=\s*)\S+', rf'\g<1>{EXTEND_TO_MINUTES}', text)
# mdu_path.write_text(text)
# print(f'TStop updated to {EXTEND_TO_MINUTES} minutes ({EXTEND_TO_MINUTES/1440:.0f} days).')
print('TStop NOT modified — run short with tracer first, then extend once CMEMS+ERA5 cover the longer period.')

## E. SWAN wave coupling

Inline version of notebook 05's config. Creates:
- `wave/Stagnone.grd` — rectangular SWAN grid ~50m (covering lagoon + western offshore)
- `wave/Stagnone.dep` — bathymetry interpolated from FM mesh
- `wave/Stagnone.mdw` — SWAN master definition
- `wave/Stagnone.swn` — SWAN input script (if using standalone)
- `dimr_config.xml` — updated to run FM + Wave with coupler
- MDU `Wavemodelnr = 3` to activate wave forces from SWAN

This is a long section — kept as a separate cell-set mirroring notebook 05 for self-containment. For brevity we **reuse** the files produced by notebook 05 if present.

In [ ]:
wave_dir = v03_dir / 'wave'
if wave_dir.exists() and (wave_dir / 'Stagnone.mdw').exists():
    print(f'Wave config found at {wave_dir} — keeping as-is.')
else:
    print('Wave config missing. Run notebook 05 first, or copy `wave/` and `dimr_config.xml` '
          'from that notebook output into v03/.')

In [ ]:
# Ensure MDU has Wavemodelnr = 3 (SWAN coupling on)
text = mdu_path.read_text()
if re.search(r'(?im)^\s*Wavemodelnr\s*=', text):
    text = re.sub(r'(?im)^(\s*Wavemodelnr\s*=\s*)\S+', r'\g<1>3', text)
else:
    text += '\n[waves]\nWavemodelnr           = 3\n'
mdu_path.write_text(text)
print('Wavemodelnr = 3 set in MDU.')

## F. EDITO entry point — `run_model.sh`

The `delft3dfm_run_docker` service on EDITO reads `run_model.sh` from the model root as its entry point (see `memory/feedback_edito_run_model_sh.md`). Copy-from-v02 brings one over, but we regenerate here so `nPart` is explicit and the file has correct LF line endings.

Also verify `dimr_config.xml` has `<process></process>` inside the FlowFM component (needed for `sed`-based process-id injection when `nPart > 1`).

## F. Verification

List v03 files; flag the critical ones present / missing.

In [ ]:
# Write run_model.sh with LF endings (never CRLF)
N_PART = 4   # EDITO quota max 8 CPUs; 4 is safe default. Set to 1 for sequential sanity runs.
MDU_NAME = 'Stagnone_dxy01_15m.mdu'

run_sh = (
    '#!/bin/bash\n'
    'set -e\n'
    f'nPart={N_PART}\n'
    'dimrFile=dimr_config.xml\n'
    'mduFolder=.\n'
    'PROCESSSTR="$(seq -s " " 0 $((nPart-1)))"\n'
    'sed -i "s/\\(<process.*>\\)[^<>]*\\(<\\/process.*\\)/\\1$PROCESSSTR\\2/" $dimrFile\n'
    f'mduFile={MDU_NAME}\n'
    '\n'
    'if [ "$nPart" == "1" ]; then\n'
    '    run_dimr.sh -m $dimrFile\n'
    'else\n'
    '    pushd $mduFolder\n'
    '        run_dflowfm.sh --partition:ndomains=$nPart:icgsolver=6 $mduFile\n'
    '    popd\n'
    '    run_dimr.sh -c $nPart -m $dimrFile\n'
    'fi\n'
)
# Binary write = no CRLF translation on Windows
(v03_dir / 'run_model.sh').write_bytes(run_sh.encode('utf-8'))
print(f'Wrote run_model.sh with nPart={N_PART} (LF endings).')

# Verify dimr_config.xml has <process></process> placeholder
dimr_path = v03_dir / 'dimr_config.xml'
dimr_text = dimr_path.read_text()
if '<process>' not in dimr_text:
    # Inject empty <process></process> after <library>dflowfm</library>
    dimr_text = dimr_text.replace(
        '<library>dflowfm</library>',
        '<library>dflowfm</library>\n    <process></process>',
        1,
    )
    dimr_path.write_text(dimr_text)
    print('Added <process></process> placeholder to dimr_config.xml')
else:
    print('dimr_config.xml already has <process> tag — no change.')

In [ ]:
critical = {
    'MDU':           'Stagnone_dxy01_15m.mdu',
    'net':           'Stagnone_dxy01_15m_net.nc',
    'pli':           'Stagnone_dxy01_15m.pli',
    'initialFields': 'initialFields.ini',
    'salinity XYZ':  'initialsalinity_hypersaline.xyz',
    'tracer XYZ':    'lagoon_tracer_init.xyz',
    'tracer BC':     'tracer_zero.bc',
    'WL BC (v02)':   'waterlevelbnd_constant_Stagnone_dxy01_15m.bc',
    'dimr_config':   'dimr_config.xml',
    'run.sh (EDITO)':'run_model.sh',
    'run.bat (local)':'run_model.bat',
}
print(f'{"label":<18s} {"present":<9s} file')
print('-' * 70)
for label, fname in critical.items():
    p = v03_dir / fname
    mark = 'YES' if p.exists() else '---'
    print(f'{label:<18s} {mark:<9s} {fname}')

In [ ]:
# Summary of v03 vs v02 changes
print('v03 changes vs v02:')
print(f'  + Initial salinity: {S_lag} psu inside lagoon, {S_off} outside (was uniform 37.5)')
print(f'  + Passive tracer "lagoon_tracer": 1 inside / 0 outside, Dirichlet=0 at open boundary')
print(f'  + SWAN wave coupling (Wavemodelnr=3) — requires wave/ + dimr_config.xml with wave component')
print(f'  + run_model.sh regenerated with nPart={N_PART}')
print(f'  - ERA5 evaporation: pending download (see section C)')
print(f'  - TStop extension for residence time: pending (see section D)')
print('\nLocal test:    cd model/dflowfm_v03 && run_model.bat')
print('EDITO run:     python scripts/edito_sync.py upload --model-dir model/dflowfm_v03')
print('               then launch delft3dfm_run_docker from Datalab UI')

## 7. Recommended run sequence

1. **Sanity run (same 9-day window as v02)** — verify tracer advecting, salinity field right, waves init clean.
2. **Inspect output** — compare BS/AE/BN water levels (waves shouldn't shift these much), check `mesh2d_lagoon_tracer` is in `_map.nc`, interior salinity ≥ 41 psu.
3. **If step 2 clean:** download extended CMEMS+ERA5 (e.g. Jul 1 – Aug 15 2025), bump `TStop`, rerun for the residence-time study → notebook 13 post-processing.
4. **If surface drift under-predicted:** tune wind drag / Charnock / Stokes drift before committing to the long run.

## 8. Known caveats

- **Tracer quantity syntax** (`initialtracer<NAME>`, `tracerbnd<NAME>`) follows the D-Flow FM 1.2.x convention. Confirm against the release notes of your installed kernel (DIMR reported `D-Flow FM 1.2.184` in v02).
- **Wave coupling** adds ~40–60% to runtime; account for that when sizing the long run.
- **Hypersaline initial field can shock the numerics** if the contrast is too sharp across a very shallow inlet. Watch for spurious overshoots in the first 12 h and widen the polygon buffer if needed.
- **`Temperature=3`** (excess/heat-flux model, already on from v02) computes net heat and latent fluxes — latent ≈ evaporation in energy terms. Check whether that already implicitly conserves salt mass before adding a separate `e` forcing (double-counting risk).
- **EDITO parallel run** requires pod CPU request ≥ `nPart` — check the Datalab UI advanced options at launch, or use the Process API to set `resources.requests.cpu`.
- **`dimr_config.xml` wave coupling** — section E reuses the file from notebook 05 if present. When v03 finally has SWAN active, the XML must declare both `<component name="DFlowFM">` and `<component name="WAVE">` plus a `<coupler>` between them. Verify before launching.